In [1]:
from wand.image import Image
from wand.drawing import Drawing
from wand.color import Color
from pathlib import Path
import pandas as pd

from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont
import tempfile

In [6]:
dir_path = Path('tornado__16SGG')

In [7]:
png_paths = sorted(list(dir_path.rglob('*.png')))
png_paths[:3]

[PosixPath('tornado__16SGG/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1A_30_v0.1.png'),
 PosixPath('tornado__16SGG/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1_30_v0.1.png'),
 PosixPath('tornado__16SGG/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1_30_v0.1/OPERA_L3_DIST-ALERT-S1_T16SGG_20240704T233950Z_20250915T180809Z_S1_30_v0.1.png')]

In [3]:
def get_date(path: Path | str):
    path = Path(path)
    name = path.stem
    date_str = name.split('_')[4]
    ts = pd.Timestamp(date_str)
    date_str_f = ts.strftime('%Y/%m/%d')
    return date_str_f

dates = [get_date(p) for p in png_paths]
dates[:3]

['2024/07/04', '2024/07/04', '2024/07/04']

In [4]:
def pngs_to_gif(png_files, output_gif):
    with Image() as gif:
        for f in png_files:
            with Image(filename=f) as frame:
                frame.delay = 100
                gif.sequence.append(frame)
        for frame in gif.sequence:
            frame.dispose = 'background'
        gif.type = 'optimize'
        gif.save(filename=output_gif)


def create_gif_with_text(png_files, output_path, text_list):
    with tempfile.TemporaryDirectory() as tmpdir:
        modified_files = []
        
        for i, (png_file, text) in enumerate(zip(png_files, text_list)):
            img = PILImage.open(png_file)
            img = img.convert('RGBA')
            draw = ImageDraw.Draw(img)
            
            font = ImageFont.truetype("Times New Roman", 150)
            
            draw.text((100, 100), text, font=font, fill=(255, 255, 255, 255), stroke_width=5, stroke_fill=(0, 0, 0, 255))
            
            temp_file = Path(tmpdir) / f'temp_frame_{i}.png'
            img.save(temp_file)
            modified_files.append(temp_file)
            img.close()
        
        with Image() as gif:
            for modified_file in modified_files:
                with Image(filename=str(modified_file)) as img:
                    img.delay = 100
                    gif.sequence.append(img)
            
            gif.type = 'optimize'
            gif.save(filename=output_path)

In [8]:
out_gif = f'{dir_path.name}__status.gif'

In [9]:
create_gif_with_text(png_paths, out_gif, dates) 